# Lesson 7A: Anomaly Detection Theory — Isolation Forest, LOF, and One-Class SVM

## Introduction

Every clustering lesson so far assumed every point belongs somewhere. Real data breaks that assumption: fraudulent transactions, sensor faults, network intrusions, manufacturing defects. These are **rare** and **unlabeled** — you rarely have a training set of "confirmed fraud" to learn from. Anomaly detection asks a different question than clustering: not "which group does this point belong to?" but "how unusual is this point relative to everything else?"

This lesson covers three genuinely different paradigms for answering that question:

1. **Isolation Forest** — isolation-based. Anomalies are *easy to isolate* with random partitioning; normal points are not.
2. **Local Outlier Factor (LOF)** — density-based. Anomalies live in regions that are *sparser than their neighbors'* neighborhoods.
3. **One-Class SVM** — boundary-based. Anomalies fall *outside a learned boundary* that encloses normal data in a (possibly kernel-transformed) feature space.

None of these require a distance metric universally — Isolation Forest needs none at all. LOF reuses the density-reachability idea from DBSCAN (Lesson 3) but turns it into a continuous score instead of a binary cluster/noise label. One-Class SVM treats the origin as the only "normal" reference point after a kernel transform.

In this lesson:
1. Frame anomaly detection as unsupervised scoring, not classification
2. Derive Isolation Forest: random partitioning, path length, anomaly score
3. Derive LOF: k-distance, reachability distance, local reachability density
4. Derive One-Class SVM: kernel trick, hyperplane separation from the origin, the nu parameter
5. Implement all three from scratch in NumPy and cross-check against scikit-learn
6. Compare the three paradigms side by side

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Framing: Anomaly Detection as Unsupervised Scoring](#framing)
4. [Isolation Forest: Theory](#iforest-theory)
5. [Isolation Forest: From Scratch](#iforest-scratch)
6. [Isolation Forest: Scikit-learn Validation](#iforest-sklearn)
7. [Local Outlier Factor: Theory](#lof-theory)
8. [Local Outlier Factor: From Scratch](#lof-scratch)
9. [Local Outlier Factor: Scikit-learn Validation](#lof-sklearn)
10. [One-Class SVM: Theory](#ocsvm-theory)
11. [One-Class SVM: From Scratch](#ocsvm-scratch)
12. [One-Class SVM: Scikit-learn Validation](#ocsvm-sklearn)
13. [Side-by-Side Comparison](#comparison)
14. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from sklearn.datasets import make_blobs
from sklearn.metrics import roc_auc_score
from typing import Tuple, List
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="framing"></a>
## Framing: Anomaly Detection as Unsupervised Scoring

### Why Not Just Classify?

Supervised classification needs labeled examples of both classes. Anomalies are rare almost by definition, so:
- Positive examples are scarce, and the ones you have may not represent *future* anomaly types
- A classifier trained on a handful of known-fraud examples overfits to that fraud pattern, missing novel ones

Unsupervised anomaly detection sidesteps this: model what "normal" looks like from the bulk of the (mostly-normal) unlabeled data, then score every point by how well it fits. No anomaly labels required at training time.

### Shared Data Setup

We'll build one synthetic dataset used across all three methods: a dense cluster of "normal" points plus a small number of injected outliers. This lets us compare paradigms on identical ground truth.

In [ ]:
# Shared synthetic dataset: normal cluster + injected outliers
X_normal, _ = make_blobs(n_samples=200, centers=1, cluster_std=1.0, center_box=(0, 0), random_state=42)

n_outliers = 20
rng = np.random.RandomState(42)
X_outliers = rng.uniform(low=-8, high=8, size=(n_outliers, 2))

X = np.vstack([X_normal, X_outliers])
y_true = np.concatenate([np.zeros(len(X_normal)), np.ones(n_outliers)])  # 1 = anomaly

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X[y_true == 0, 0], X[y_true == 0, 1], s=50, alpha=0.6, edgecolors='k', label=f'Normal (n={len(X_normal)})')
ax.scatter(X[y_true == 1, 0], X[y_true == 1, 1], s=100, c='red', marker='x', linewidth=2, label=f'Injected outliers (n={n_outliers})')
ax.set_title('Shared Dataset: Normal Cluster + Injected Outliers', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total points: {len(X)}, contamination: {n_outliers/len(X):.1%}")

<a name="iforest-theory"></a>
## Isolation Forest: Theory

### Core Insight

Instead of profiling normal points and flagging what deviates, **isolate** every point directly: pick a random feature, pick a random split value between that feature's min and max, and partition the data. Repeat recursively.

**Anomalies are easier to isolate than normal points.** A point in a dense cluster needs many random splits before it's alone in its own partition — it's surrounded by similar neighbors. A point far from everything else gets separated in just one or two splits, because almost any random cut isolates it.

### Isolation Trees

An **isolation tree (iTree)** is a binary tree built by recursive random partitioning:
1. If the current node has 1 point (or reaches a height limit), stop — it's a leaf.
2. Otherwise pick a random feature $q$ and a random split value $p$ between $\min(q)$ and $\max(q)$ in the current node's data.
3. Recurse into the two child partitions.

The **path length** $h(x)$ of a point $x$ is the number of edges from the root to the leaf containing $x$.

### Path Length as an Anomaly Signal

Across an ensemble of $t$ randomly-built isolation trees, average the path length $E[h(x)]$ for each point. Short average path length $\Rightarrow$ easy to isolate $\Rightarrow$ likely anomalous.

### Normalizing the Score

Raw path length depends on sample size, so it's normalized against $c(n)$, the average path length of an unsuccessful search in a Binary Search Tree of $n$ points:

$$c(n) = 2H(n-1) - \frac{2(n-1)}{n}, \qquad H(i) \approx \ln(i) + 0.5772156649 \text{ (Euler-Mascheroni constant)}$$

The anomaly score for point $x$ is:

$$s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}$$

Interpretation:
- $s \to 1$: $E[h(x)] \to 0$ — isolated almost immediately, **strong anomaly**
- $s \to 0.5$: $E[h(x)] \approx c(n)$ — typical path length, **normal**
- $s \to 0$: $E[h(x)] \to n-1$ — very hard to isolate (rarely happens in practice), **very normal**

No distance metric, no density estimate — just how many random cuts it takes to separate a point from everything else. This makes Isolation Forest $O(n \log n)$ and effective in high dimensions where distance-based methods struggle (curse of dimensionality).

<a name="iforest-scratch"></a>
## Isolation Forest: From Scratch

In [ ]:
class IsolationTreeNode:
    """A single node in an isolation tree: internal split or leaf."""
    def __init__(self):
        self.feature = None
        self.split_value = None
        self.left = None
        self.right = None
        self.size = 0        # points that reached this node (leaves only need this)
        self.is_leaf = False


def build_itree(X: NDArray, height: int, height_limit: int) -> IsolationTreeNode:
    """Recursively build one isolation tree via random feature + random split."""
    node = IsolationTreeNode()
    node.size = len(X)

    if height >= height_limit or len(X) <= 1:
        node.is_leaf = True
        return node

    n_features = X.shape[1]
    feature = np.random.randint(n_features)
    col = X[:, feature]

    if col.min() == col.max():
        node.is_leaf = True
        return node

    split_value = np.random.uniform(col.min(), col.max())
    node.feature = feature
    node.split_value = split_value

    left_mask = col < split_value
    node.left = build_itree(X[left_mask], height + 1, height_limit)
    node.right = build_itree(X[~left_mask], height + 1, height_limit)
    return node


def path_length(x: NDArray, node: IsolationTreeNode, height: int = 0) -> float:
    """Path length for a single point, with a size-based correction at early leaves."""
    if node.is_leaf:
        return height + c_factor(node.size)
    if x[node.feature] < node.split_value:
        return path_length(x, node.left, height + 1)
    else:
        return path_length(x, node.right, height + 1)


def c_factor(n: int) -> float:
    """Average path length of an unsuccessful BST search over n points."""
    if n <= 1:
        return 0.0
    euler_gamma = 0.5772156649
    harmonic = np.log(n - 1) + euler_gamma
    return 2 * harmonic - 2 * (n - 1) / n


class IsolationForestScratch:
    """Ensemble of isolation trees producing normalized anomaly scores."""
    def __init__(self, n_trees: int = 100, sample_size: int = 256, random_state: int = 42):
        self.n_trees = n_trees
        self.sample_size = sample_size
        self.random_state = random_state
        self.trees: List[IsolationTreeNode] = []

    def fit(self, X: NDArray) -> "IsolationForestScratch":
        np.random.seed(self.random_state)
        n = min(self.sample_size, len(X))
        height_limit = int(np.ceil(np.log2(n))) if n > 1 else 1
        self.trees = []
        for _ in range(self.n_trees):
            idx = np.random.choice(len(X), size=n, replace=False)
            self.trees.append(build_itree(X[idx], height=0, height_limit=height_limit))
        self._c_n = c_factor(n)
        return self

    def score_samples(self, X: NDArray) -> NDArray:
        """Anomaly score in [0, 1]; higher = more anomalous."""
        scores = np.zeros(len(X))
        for i, x in enumerate(X):
            avg_path = np.mean([path_length(x, tree) for tree in self.trees])
            scores[i] = 2 ** (-avg_path / self._c_n)
        return scores


iforest_scratch = IsolationForestScratch(n_trees=100, sample_size=256, random_state=42).fit(X)
scores_scratch = iforest_scratch.score_samples(X)

print(f"From-scratch Isolation Forest:")
print(f"  Mean score (normal):  {scores_scratch[y_true == 0].mean():.3f}")
print(f"  Mean score (outlier): {scores_scratch[y_true == 1].mean():.3f}")
print(f"  ROC-AUC vs injected outliers: {roc_auc_score(y_true, scores_scratch):.3f}")

<a name="iforest-sklearn"></a>
## Isolation Forest: Scikit-learn Validation

In [ ]:
iforest_sk = IsolationForest(n_estimators=100, max_samples=min(256, len(X)), contamination=n_outliers/len(X), random_state=42)
iforest_sk.fit(X)
scores_sk = -iforest_sk.score_samples(X)  # sklearn returns higher = more normal; flip sign to match our convention
labels_sk = iforest_sk.predict(X)  # -1 = anomaly, 1 = normal

print(f"Scikit-learn IsolationForest:")
print(f"  ROC-AUC vs injected outliers: {roc_auc_score(y_true, scores_sk):.3f}")
print(f"  Flagged anomalies: {(labels_sk == -1).sum()} (of {n_outliers} injected)")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, scores, title in zip(axes, [scores_scratch, scores_sk], ['From-Scratch iForest', 'Scikit-learn IsolationForest']):
    sc = ax.scatter(X[:, 0], X[:, 1], c=scores, cmap='RdYlBu_r', s=60, edgecolors='k', linewidth=0.5)
    plt.colorbar(sc, ax=ax, label='Anomaly score')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Both agree: injected outliers (isolated corners) score highest, the dense cluster scores lowest")

<a name="lof-theory"></a>
## Local Outlier Factor: Theory

### Core Insight

Isolation Forest asks "how easy is this point to isolate globally?" LOF asks a **local** question instead: "is this point's neighborhood sparser than its neighbors' neighborhoods?" This matters when clusters have different densities — a point can be perfectly normal for a sparse cluster but anomalous for a dense one, and a single global density threshold (like DBSCAN's ε) can't capture that. LOF adapts the notion of density to each point's local neighborhood.

### K-Distance and K-Neighborhood

The **k-distance** of point $p$, $\text{k-dist}(p)$, is the distance to its $k$-th nearest neighbor. The **k-neighborhood** $N_k(p)$ is the set of all points within that distance (may exceed $k$ points if there are ties).

### Reachability Distance

To reduce statistical noise for points very close to each other, LOF smooths raw distance with:

$$\text{reach-dist}_k(p, o) = \max(\text{k-dist}(o), \, d(p, o))$$

This is deliberately asymmetric: the reachability distance *from* $p$ *to* $o$ is measured relative to $o$'s own k-distance, not $p$'s.

### Local Reachability Density (LRD)

The LRD of $p$ is the inverse of the average reachability distance from $p$ to its neighbors:

$$\text{lrd}_k(p) = \left( \frac{\sum_{o \in N_k(p)} \text{reach-dist}_k(p, o)}{|N_k(p)|} \right)^{-1}$$

High LRD means $p$ sits in a dense region (its neighbors are reachable at short distance). Low LRD means $p$ is in a sparse region.

### LOF Score

Finally, compare $p$'s density to the *average density of its neighbors*:

$$\text{LOF}_k(p) = \frac{\sum_{o \in N_k(p)} \frac{\text{lrd}_k(o)}{\text{lrd}_k(p)}}{|N_k(p)|}$$

Interpretation:
- $\text{LOF}(p) \approx 1$: $p$'s density matches its neighbors' — **normal**, comparable local density
- $\text{LOF}(p) \gg 1$: $p$'s density is much lower than its neighbors' — **outlier**, sits in a locally sparse pocket
- $\text{LOF}(p) < 1$: $p$ is in a denser region than its neighbors — unusual in the other direction, but not typically flagged

This is exactly the density-reachability concept from DBSCAN (Lesson 3), but instead of thresholding it into core/border/noise, LOF turns it into a continuous, per-point, locally-relative score.

<a name="lof-scratch"></a>
## Local Outlier Factor: From Scratch

In [ ]:
def k_distance_and_neighbors(X: NDArray, k: int) -> Tuple[NDArray, List[NDArray]]:
    """k-distance and k-neighborhood indices for every point."""
    distances = cdist(X, X)
    np.fill_diagonal(distances, np.inf)  # exclude self

    k_dist = np.zeros(len(X))
    neighbor_idx = []
    for i in range(len(X)):
        order = np.argsort(distances[i])
        k_dist[i] = distances[i, order[k - 1]]
        # include ties at the k-th distance, matching the k-neighborhood definition
        neighbors = order[distances[i, order] <= k_dist[i]]
        neighbor_idx.append(neighbors)
    return k_dist, neighbor_idx


def lof_scratch(X: NDArray, k: int = 10) -> NDArray:
    """Local Outlier Factor for every point in X."""
    distances = cdist(X, X)
    np.fill_diagonal(distances, np.inf)
    k_dist, neighbor_idx = k_distance_and_neighbors(X, k)

    n = len(X)
    lrd = np.zeros(n)
    for p in range(n):
        neighbors = neighbor_idx[p]
        reach_dists = np.maximum(k_dist[neighbors], distances[p, neighbors])
        lrd[p] = 1.0 / (reach_dists.mean() + 1e-12)

    lof = np.zeros(n)
    for p in range(n):
        neighbors = neighbor_idx[p]
        lof[p] = (lrd[neighbors] / lrd[p]).mean()

    return lof


lof_scores_scratch = lof_scratch(X, k=10)

print(f"From-scratch LOF:")
print(f"  Mean LOF (normal):  {lof_scores_scratch[y_true == 0].mean():.3f}")
print(f"  Mean LOF (outlier): {lof_scores_scratch[y_true == 1].mean():.3f}")
print(f"  ROC-AUC vs injected outliers: {roc_auc_score(y_true, lof_scores_scratch):.3f}")

<a name="lof-sklearn"></a>
## Local Outlier Factor: Scikit-learn Validation

In [ ]:
lof_sk = LocalOutlierFactor(n_neighbors=10, contamination=n_outliers/len(X))
labels_lof_sk = lof_sk.fit_predict(X)  # -1 = anomaly, 1 = normal
lof_scores_sk = -lof_sk.negative_outlier_factor_  # sklearn stores the negated LOF; flip back

print(f"Scikit-learn LocalOutlierFactor:")
print(f"  ROC-AUC vs injected outliers: {roc_auc_score(y_true, lof_scores_sk):.3f}")
print(f"  Flagged anomalies: {(labels_lof_sk == -1).sum()} (of {n_outliers} injected)")
print(f"\nCorrelation between from-scratch and sklearn LOF scores: {np.corrcoef(lof_scores_scratch, lof_scores_sk)[0, 1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, scores, title in zip(axes, [lof_scores_scratch, lof_scores_sk], ['From-Scratch LOF', 'Scikit-learn LocalOutlierFactor']):
    sc = ax.scatter(X[:, 0], X[:, 1], c=scores, cmap='RdYlBu_r', s=60, edgecolors='k', linewidth=0.5, vmax=np.percentile(scores, 98))
    plt.colorbar(sc, ax=ax, label='LOF score')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 LOF > 1 means locally sparser than neighbors — exactly the injected outliers")

<a name="ocsvm-theory"></a>
## One-Class SVM: Theory

### Core Insight

One-Class SVM (Schölkopf et al., 2001) reframes anomaly detection as a **boundary-fitting** problem: find the smallest region in feature space that contains most of the data, then flag anything outside it. Rather than separating two classes like a standard SVM, it separates the data from **the origin** — after mapping into a (typically RBF) kernel feature space where the origin becomes a meaningful "background" reference point.

### The Primal Problem

Given data mapped into feature space via $\phi(x)$, find a hyperplane $w^T \phi(x) = \rho$ that separates the data from the origin with maximum margin, allowing a controlled fraction of points to fall on the wrong side (slack $\xi_i$):

$$\min_{w, \xi, \rho} \ \frac{1}{2}\|w\|^2 + \frac{1}{\nu n}\sum_{i=1}^n \xi_i - \rho$$
$$\text{subject to } \ w^T \phi(x_i) \geq \rho - \xi_i, \quad \xi_i \geq 0$$

### The Nu Parameter

$\nu \in (0, 1]$ is simultaneously:
- An **upper bound** on the fraction of training points allowed to be outliers (margin violations)
- A **lower bound** on the fraction of points that become support vectors

Set $\nu$ close to your expected contamination rate — it's the direct knob for how strict the boundary is.

### The Dual Problem (Kernel Trick)

As with standard SVMs, the dual formulation only needs inner products $\phi(x_i)^T\phi(x_j) = K(x_i, x_j)$, letting us use a kernel (typically RBF, $K(x_i, x_j) = \exp(-\gamma\|x_i - x_j\|^2)$) without ever computing $\phi(x)$ explicitly:

$$\min_{\alpha} \ \frac{1}{2}\sum_{i,j}\alpha_i \alpha_j K(x_i, x_j)$$
$$\text{subject to } \ 0 \leq \alpha_i \leq \frac{1}{\nu n}, \quad \sum_i \alpha_i = 1$$

Points with $0 < \alpha_i < \frac{1}{\nu n}$ are support vectors on the margin; points with $\alpha_i = \frac{1}{\nu n}$ are margin violations (potential outliers).

### Decision Function

$$f(x) = \sum_i \alpha_i K(x_i, x) - \rho$$

$f(x) < 0 \Rightarrow$ **anomaly** (outside the learned boundary). $\rho$ is recovered from any support vector strictly on the margin ($0 < \alpha_i < \frac{1}{\nu n}$), where $f(x_i) = 0$ exactly.

Where Isolation Forest needs no distance and LOF needs Euclidean k-nearest-neighbor distances, One-Class SVM needs a kernel and is the most sensitive to hyperparameter choice ($\nu$, $\gamma$) of the three.

<a name="ocsvm-scratch"></a>
## One-Class SVM: From Scratch

Solving the dual QP exactly requires an SMO-style solver. For a from-scratch illustration at this scale, **projected gradient descent** on the dual objective converges to the same solution: at each step, take a gradient step on $\alpha$, then project back onto the feasible set ($0 \le \alpha_i \le \frac{1}{\nu n}$, $\sum \alpha_i = 1$ — a *capped simplex*).

The projection itself needs care: naively clipping to $[0, \frac{1}{\nu n}]$ and then rescaling to restore $\sum \alpha_i = 1$ can push values back outside the cap, so it never actually lands on the feasible set. The correct Euclidean projection onto a capped simplex solves for a single shift $\lambda$ such that $\sum_i \text{clip}(v_i - \lambda, 0, C) = 1$ — found by bisection, since the sum is monotone non-increasing in $\lambda$.

In [ ]:
def rbf_kernel(X1: NDArray, X2: NDArray, gamma: float) -> NDArray:
    sq_dists = cdist(X1, X2, metric='sqeuclidean')
    return np.exp(-gamma * sq_dists)


def project_capped_simplex(v: NDArray, upper_bound: float, n_bisect: int = 100) -> NDArray:
    """Euclidean projection of v onto {0 <= alpha_i <= upper_bound, sum(alpha) = 1}."""
    lo, hi = v.min() - 1.0, v.max()
    for _ in range(n_bisect):
        mid = (lo + hi) / 2
        total = np.clip(v - mid, 0, upper_bound).sum()
        if total > 1:
            lo = mid
        else:
            hi = mid
    return np.clip(v - mid, 0, upper_bound)


class OneClassSVMScratch:
    """One-Class SVM dual solved via projected gradient descent (capped-simplex projection)."""
    def __init__(self, nu: float = 0.1, gamma: float = 0.5, n_iter: int = 3000):
        self.nu = nu
        self.gamma = gamma
        self.n_iter = n_iter

    def fit(self, X: NDArray) -> "OneClassSVMScratch":
        n = len(X)
        self.X_train = X
        K = rbf_kernel(X, X, self.gamma)
        upper_bound = 1.0 / (self.nu * n)

        # Step size 1/L, L = largest eigenvalue of K, guarantees descent on the quadratic (1/2) alpha^T K alpha
        lr = 1.0 / np.linalg.eigvalsh(K)[-1]

        alpha = np.full(n, 1.0 / n)  # start feasible: sums to 1
        for _ in range(self.n_iter):
            grad = K @ alpha  # gradient of (1/2) alpha^T K alpha
            alpha = project_capped_simplex(alpha - lr * grad, upper_bound)

        self.alpha = alpha
        self.upper_bound = upper_bound

        # rho from a free support vector (0 < alpha_i < upper_bound), where f(x_i) = 0
        margin_sv = np.where((alpha > 1e-6) & (alpha < upper_bound - 1e-6))[0]
        if len(margin_sv) == 0:
            margin_sv = np.where(alpha > 1e-6)[0]  # fallback if none strictly on the margin
        self.rho = np.mean(K[margin_sv] @ alpha)
        return self

    def decision_function(self, X: NDArray) -> NDArray:
        K = rbf_kernel(X, self.X_train, self.gamma)
        return K @ self.alpha - self.rho

    def predict(self, X: NDArray) -> NDArray:
        return np.where(self.decision_function(X) >= 0, 1, -1)


ocsvm_scratch = OneClassSVMScratch(nu=0.1, gamma=0.05, n_iter=3000).fit(X)
decision_scratch = ocsvm_scratch.decision_function(X)
anomaly_scores_scratch = -decision_scratch  # flip so higher = more anomalous, matching iForest/LOF convention

print(f"From-scratch One-Class SVM:")
print(f"  Support vectors: {(ocsvm_scratch.alpha > 1e-6).sum()} of {len(X)}")
print(f"  ROC-AUC vs injected outliers: {roc_auc_score(y_true, anomaly_scores_scratch):.3f}")

<a name="ocsvm-sklearn"></a>
## One-Class SVM: Scikit-learn Validation

In [ ]:
ocsvm_sk = OneClassSVM(kernel='rbf', nu=0.1, gamma=0.05)
ocsvm_sk.fit(X)
decision_sk = ocsvm_sk.decision_function(X)
anomaly_scores_sk = -decision_sk
labels_ocsvm_sk = ocsvm_sk.predict(X)  # -1 = anomaly, 1 = normal

print(f"Scikit-learn OneClassSVM:")
print(f"  Support vectors: {len(ocsvm_sk.support_)} of {len(X)}")
print(f"  ROC-AUC vs injected outliers: {roc_auc_score(y_true, anomaly_scores_sk):.3f}")
print(f"  Flagged anomalies: {(labels_ocsvm_sk == -1).sum()} (of {n_outliers} injected)")
print(f"\nCorrelation between from-scratch and sklearn decision scores: {np.corrcoef(decision_scratch, decision_sk)[0, 1]:.3f}")

# Decision boundary for the sklearn model (ground truth for the region shape)
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
                      np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200))
Z = ocsvm_sk.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, scores, title in zip(axes, [anomaly_scores_scratch, anomaly_scores_sk], ['From-Scratch One-Class SVM', 'Scikit-learn OneClassSVM']):
    ax.contourf(xx, yy, Z, levels=[Z.min(), 0, Z.max()], colors=['#fddede', '#ffffff'], alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black', linestyles='dashed')
    sc = ax.scatter(X[:, 0], X[:, 1], c=scores, cmap='RdYlBu_r', s=60, edgecolors='k', linewidth=0.5)
    plt.colorbar(sc, ax=ax, label='Anomaly score')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 The dashed line is the learned boundary (sklearn); points outside it score as anomalies in both implementations")

<a name="comparison"></a>
## Side-by-Side Comparison

| | Isolation Forest | Local Outlier Factor | One-Class SVM |
|---|---|---|---|
| **Paradigm** | Isolation-based | Density-based (local) | Boundary-based |
| **Needs a distance metric?** | No | Yes (k-NN) | Yes (kernel) |
| **Key hyperparameters** | `n_estimators`, `max_samples` | `n_neighbors` (k) | `nu`, `gamma`, kernel |
| **Training complexity** | $O(t \cdot \psi \log \psi)$ — sub-linear in $n$ via subsampling | $O(n^2)$ naive, $O(n \log n)$ with spatial index | $O(n^2)$ to $O(n^3)$ (QP) |
| **Scoring complexity** | $O(t \log \psi)$ per point | $O(n)$ per point (or $O(\log n)$ indexed) | $O(n_{sv})$ per point |
| **Handles varying density clusters** | Weaker — global isolation criterion | **Strong** — explicitly local/relative | Weaker — one global boundary |
| **High-dimensional data** | **Strong** — no distance/density curse | Weak — k-NN distances degrade | Moderate — kernel choice matters |
| **Interpretability** | Path length is intuitive | LRD ratio is intuitive | Margin/boundary is intuitive but kernel-space is abstract |
| **Novel-point scoring (no retrain)** | Fast (walk existing trees) | Requires recomputing local neighborhood | Fast (evaluate kernel against support vectors) |
| **Sensitive to hyperparameters** | Least | Moderate (k changes locality) | Most ($\nu$, $\gamma$ both matter) |

### When to Use Which

- **Isolation Forest**: default first choice for tabular data, especially high-dimensional or large-scale (subsampling keeps it fast); weakest at handling clusters of very different densities in the same dataset.
- **Local Outlier Factor**: best when normal data forms clusters of noticeably different densities — its whole design point is relative, local comparison.
- **One-Class SVM**: best when you can afford to tune $\nu$ and $\gamma$ carefully and want an explicit geometric boundary; slowest to train on larger datasets, most hyperparameter-sensitive.

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **Anomaly detection is unsupervised scoring, not classification** — model what's normal, score deviation from it, no anomaly labels needed for training.
2. **Isolation Forest** isolates points via random partitioning; short average path length across an ensemble of trees signals an anomaly. No distance metric required, sub-linear training via subsampling.
3. **Local Outlier Factor** compares each point's local density (via reachability distance) to its neighbors' — the same density-reachability idea as DBSCAN, turned into a continuous, locally-relative score.
4. **One-Class SVM** learns a boundary separating the data from the origin in kernel feature space; $\nu$ directly controls the expected outlier fraction.
5. **The three paradigms fail differently** — isolation and boundary methods assume roughly uniform density; LOF is built precisely for the case where they don't.

### When to Use Anomaly Detection at All

- ✅ Rare, unlabeled, or unpredictable-shape deviations (fraud, faults, intrusions)
- ✅ No reliable labeled anomaly training set
- ❌ When you *do* have enough labeled examples of the anomaly class — supervised classification will usually outperform

### Next Steps

In Lesson 7B (practical), we'll:
- Apply all three methods to a fraud-detection-style dataset
- Build a PyTorch autoencoder as a fourth (reconstruction-based) anomaly detector
- Compare precision/recall trade-offs at different contamination thresholds
- Discuss threshold selection without labeled validation data

You now have three genuinely different lenses for spotting the unusual:
1. **Isolation Forest**: how few random cuts does it take to isolate this point?
2. **LOF**: is this point's neighborhood sparser than its neighbors' neighborhoods?
3. **One-Class SVM**: does this point fall outside the learned normal-region boundary?

Pick the lens that matches your data's structure 🎯